# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading and exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All data elements—record sets, fields, columns—are referenced by their `@id` attributes as specified by the Croissant schema.

### Dataset Source
The dataset Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata and print basic info
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}\n")

## 2. Data Overview
Review all available record sets (tables), their `@id`s, and their fields. This lists the structure of the data you can explore. **All `@id`s within the Croissant schema are shown explicitly for precise referencing.**

In [ ]:
# List all record sets, their @ids, fields, and columns
print("Available Record Sets (by @id):\n-------------------------------")
for rs in metadata.record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    print("")

### Preview a Record Set
Let's preview the first few records (rows) from a main record set. The main table in this dataset is usually mass spectrometry protein quantification. **Replace the `main_record_set_id` below with the correct `@id` from above if needed.**

In [ ]:
# Choose a main record set id (replace with @id from output above)
main_record_set_id = metadata.record_sets[0].id  # or manually set, e.g. 'cr:MassSpectrometryProteins'
print(f"Showing records for record set @id: {main_record_set_id}\n")

for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Load all data tables into DataFrames for analysis. Use `@id` for record sets. The table and column names can be found in the previous overview step.

In [ ]:
# Load all record sets into DataFrames, referencing them by @id
dataframes = {}
for rs in metadata.record_sets:
    recs = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(recs)

# Print columns for the main record set
print(f"Columns for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data, referencing columns and fields by their `@id`.

For this example, we'll:
- Choose a numeric field such as "protein_coverage_percent" or "abundance" (set via its field `@id` as shown in section 2)
- Filter for rows above a threshold
- Normalize the selected field
- Optionally group by another field, such as 'sample' or 'protein_class', if available

In [ ]:
# Set the numeric field and group field by @id as found in the printouts above
# For demonstration, common fields might be:
# 'cr:protein_coverage_percent', 'cr:abundance', 'cr:peptide_count', 'cr:sample_id', 'cr:protein_class', 'cr:normalized_abundance'

numeric_field_id = None
possible_numeric_fields = [col for col in dataframes[main_record_set_id].columns if 'abundance' in col or 'coverage' in col or 'peptide' in col or 'count' in col]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # pick first

if numeric_field_id is None:
    raise RuntimeError("Couldn't automatically find a numeric field. Please set 'numeric_field_id' to an appropriate column '@id'.")
print(f"Using numeric field (by @id): {numeric_field_id}")

threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field if present
group_field_candidates = [col for col in dataframes[main_record_set_id].columns if 'class' in col or 'type' in col or 'sample' in col]
group_field_id = group_field_candidates[0] if group_field_candidates else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped mean values by {group_field_id} (field '@id'):")
    print(grouped_df.head())

## 5. Visualization
Below we visualize the numeric field distribution and, if possible, its relation to a grouping field.

In [ ]:
import matplotlib.pyplot as plt

# Distribution histogram
filtered_df[numeric_field_id].hist(bins=30, figsize=(7,4))
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group field if present
if group_field_id:
    plt.figure(figsize=(10,5))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We've explored the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`. The dataset's structure, fields, and relationships were navigated entirely using Croissant `@id` references, ensuring accuracy and reproducibility. Further analysis can be conducted by examining additional fields or aggregating results across experimental conditions.